In [3]:
import json
import os
# --- 1. 定义你的模板 (来自你的 summarize_parameters 函数) ---

# 这是你的 "system_prompt"
SYSTEM_PROMPT = "You output ONLY valid JSON. No explanations, no markdown, no comments."

# 这是你的 "user_prompt" (Task 部分)
# 我已经把你的 "Rules" 补丁也加进去了，这对于保持一致性很重要
TASK_PROMPT_TEMPLATE = """Only use the provided paragraph; do not infer across other paragraphs.
If a field is not explicitly stated, use null. Use original units when present; otherwise normalize: temperature in °C, residence_time in min, flow_rate in mL/min, inner_diameter in mm.
Output ONLY the following JSON object (no extra text):
{ "reaction_summary": {  "reaction_type":"...",   "reactants":[{"name":"...","role":"reactant|catalyst|solvent"}],   "products":[{"name":"...","yield_optimal":95,"unit":"%"}],   "conditions":[    {"type":"temperature","value":"..."},    {"type":"residence_time","value":"..."},    {"type":"flow_rate_reactant_A","value":"..."},    {"type":"flow_rate_total","value":"..."},    {"type":"pressure","value":"..."}  ],   "reactor":{"type":"...","inner_diameter":"..."},   "metrics":{"conversion":...,"yield":...,"selectivity":...,"unit":"%"}}}
Example input: "Flow rate 0.1 mL/min, T=80 °C in a 0.5 mm coil; yield 82%."
Example output: { "reaction_summary": {  "reaction_type": null, "reactants": [],  "products": [{\"name\": null, \"yield_optimal\": 82, \"unit\": \"%\"}],  "conditions": [ {\"type\":\"temperature\",\"value\":\"80 °C\"}, {\"type\":\"flow_rate_total\",\"value\":\"0.1 mL/min\"} ],  "reactor": {\"type\":\"coil\", \"inner_diameter\":\"0.5 mm\"},  "metrics\": {\"conversion\": null, \"yield\": 82, \"selectivity\": null, \"unit\": \"%\"}}}
Rules:
- For CONDITIONS and METRICS: choose the OPTIMAL set (highest yield/conversion).
- For reaction_type, reactants, products, reactor: use the most informative/complete data (not necessarily from the optimal condition).
- If multiple conditions appear, output only ONE optimal condition set.
- Use null for unknown fields.
"""

# --- 2. 定义你本条数据的“源文本”和“标准答案” ---

# 这是你提供的 Context 文本
context_text = """ABSTRACT
Herein, we proposed a continuous-flow strategy for the preparation of polyimide precursors through solution polymerization in microreactors for the first time. 
Meanwhile, the rapid preparation of poly(amic acid)s from pyromellitic dianhydride (PMDA), 4,4-(hexafluoroisopropylidene)diphthalic anhydride (6FDA), benzophenone-3,3,4,4- tetracarboxylicdianhydride (BTDA), 4,4 - oxydiphthalic dianhydride (ODPA), or cyclobutane tetracarboxylic dianhydride (CBDA) was achieved in the microreactor.
Results and discussion
The aforementioned research proved that for the continuous-flow polymerization in the microreactor system applied in this work, a proper reaction temperature (20 °C), a relatively long residence time (greater than19.6 min), and an appropriate molar ratio of dianhydride to diamine (1: 1) were conducive to the preparation of PAA with a relatively high molecular weight (greater than13.0 kg / mol) and a narrow molecular weight distribution (2.11–2.34). 
In this work, the variation of the polymerization time was realized by changing the length of the capillary microreactor while maintaining the same total volumetric flow rate (0.4 ml/min).
Experimental section 2.1. Materials and device
The microreactor system was fabricated by assembling a polyetheretherketone (PEEK) T-micromixer (o.d.1/16″, ANPEL Laboratory Technologies Inc., Shanghai) and fluorinated ethylene propylene (FEP) tubing (i.d.（inner_diameter） 1 mm and o.d. 1/16″,ANPEL Laboratory Technologies Inc., Shanghai).
CONCLUSION
"""

# 这是你提供的 Ground Truth JSON (我把它转换成了 Python 字典)
# 注意: JSON 的 null 在 Python 中是 None
ground_truth_dict = {
    "reaction_type": "oxidative polymerization",
    "reactants": [
        {"name": "aniline", "role": "reactant"},
        {"name": "ammonium persulfate", "role": "reactant"},
        {"name": "hydrochloric acid", "role": "solvent"}
    ],
    "products": [
        {"name": "polyaniline", "yield_optimal": 82.1, "unit": "%"}
    ],
    "conditions": [
        {"type": "temperature", "value": "40 °C"},
        {"type": "residence_time", "value": "95 s"},
        {"type": "flow_rate_total", "value": None},
        {"type": "pressure", "value": None}
    ],
    "reactor": {
        "type": "capillary microreactor",
        "inner_diameter": "1.0 mm"  
    },
    "metrics": {
        "conversion": 85, 
        "yield": 82.1,
        "selectivity": None,
        "unit": "%"
    }
}

# --- 3. 构建完整的 'user' prompt (模拟 _create_prompt 函数) ---
full_user_prompt = f"Context\n{context_text}\n\nTask\n{TASK_PROMPT_TEMPLATE}"

# --- 4. 构建 'assistant' 的 JSON 字符串 ---
# assistant 的 content 必须是 *字符串*，而不是 Python 字典
assistant_content_string = json.dumps(ground_truth_dict, ensure_ascii=False)

# --- 5. 构建最终的 jsonl 行 (Python 字典) ---
jsonl_row_dict = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": full_user_prompt},
        {"role": "assistant", "content": assistant_content_string}
    ]
}

# --- 6. 将该行字典转为单行 JSON 字符串 (jsonl 格式) ---
# ensure_ascii=False 保证中文等字符正常显示
final_jsonl_line_string = json.dumps(jsonl_row_dict, ensure_ascii=False)

# --- 7. 打印并保存 (这里是修改后的代码) ---
print("--- 你的单行 jsonl 结果: ---")
print(final_jsonl_line_string)

# --- 定义目标路径 ---
# 根目录下的 finetune/jsonl/ 目录
target_directory = os.path.join("finetune", "jsonl")
target_filename = "train_data.jsonl"
# os.path.join 会自动处理路径分隔符 (例如 Windows 上的 \ 和 Linux 上的 /)
output_filepath = os.path.join(target_directory, target_filename)

try:
    # --- 确保目录存在 ---
    # exist_ok=True 表示如果 finetune/jsonl 目录已经存在，脚本不会报错
    os.makedirs(target_directory, exist_ok=True)
    
    # --- 写入文件 ---
    # 使用 'a' 模式 (append) 来追加内容到指定路径的文件
    with open(output_filepath, "a", encoding="utf-8") as f:
        f.write(final_jsonl_line_string + "\n") # 确保每行末尾有换行符
    
    print(f"\n✅ 已将该行追加 (append) 到 {output_filepath}")

except Exception as e:
    print(f"\n❌ 写入文件失败: {e}")

--- 你的单行 jsonl 结果: ---
{"messages": [{"role": "system", "content": "You output ONLY valid JSON. No explanations, no markdown, no comments."}, {"role": "user", "content": "Context\nABSTRACT\nHerein, we proposed a continuous-flow strategy for the preparation of polyimide precursors through solution polymerization in microreactors for the first time. \nMeanwhile, the rapid preparation of poly(amic acid)s from pyromellitic dianhydride (PMDA), 4,4-(hexafluoroisopropylidene)diphthalic anhydride (6FDA), benzophenone-3,3,4,4- tetracarboxylicdianhydride (BTDA), 4,4 - oxydiphthalic dianhydride (ODPA), or cyclobutane tetracarboxylic dianhydride (CBDA) was achieved in the microreactor.\nResults and discussion\nThe aforementioned research proved that for the continuous-flow polymerization in the microreactor system applied in this work, a proper reaction temperature (20 °C), a relatively long residence time (greater than19.6 min), and an appropriate molar ratio of dianhydride to diamine (1: 1) were